In [ ]:
import os
import pickle
from tqdm.notebook import tqdm
import json

from medcat.cat import CAT
from medcat.utils.preprocess_snomed import get_all_children

import pandas as pd
import numpy as np
import re
import random

from datetime import date

_rel_path = os.path.join("..", "..", "..", "..")
base_path = os.path.abspath(_rel_path)

project_path = os.path.join(base_path, "projects", "M-46-G_chronic_kidney_disease")

data_path = os.path.join(project_path, "data")

annotation_path_base = os.path.join(data_path, "processed_data", "ann_folder_path", "20250719_ckd_comorbidities_search")
annotation_path_upd = os.path.join(data_path, "processed_data", "ann_folder_path", "20251120_ckd_comorbidities_search")
annotation_path_final = os.path.join(data_path, "processed_data", "ann_folder_path", "20260203_ckd_comorbidities_search")

## Format MedCAT output

In [ ]:
model_dir = 'models/20241120_trained_ckd_model'
modelpack = 'medcat_model.zip'

model_pack_path = os.path.join(base_path, model_dir, modelpack)

In [ ]:
cat = CAT.load_model_pack(model_pack_path)

In [ ]:
annotation_base_chunks = os.listdir(annotation_path_base)
annotation_base_chunks.remove('annotated_ids.pickle')

In [ ]:
annotations = {}
for file in annotation_base_chunks:
    with open(os.path.join(annotation_path_base, file), 'rb') as pkl_file:
              annotations.update(pickle.load(pkl_file))
            
print(f'Total Annotations: {len(annotations)}')

In [ ]:
annotation_upd_chunks = os.listdir(annotation_path_upd)
annotation_upd_chunks.remove('annotated_ids.pickle')

In [ ]:
for file in annotation_upd_chunks:
    with open(os.path.join(annotation_path_upd, file), 'rb') as pkl_file:
              annotations.update(pickle.load(pkl_file))
            
print(f'Total Annotations: {len(annotations)}')

In [ ]:
annotation_final_chunks = os.listdir(annotation_path_final)
annotation_final_chunks.remove('annotated_ids.pickle')

In [ ]:
for file in annotation_final_chunks:
    with open(os.path.join(annotation_path_final, file), 'rb') as pkl_file:
              annotations.update(pickle.load(pkl_file))
            
print(f'Total Annotations: {len(annotations)}')

In [ ]:
relevant_annotations = {}
for doc, anns in annotations.items():
    doc_annotations = []
    for i, ann in anns['entities'].items():
        if (ann['meta_anns']['Subject']['value'] == 'Patient' and
            ann['meta_anns']['Presence']['value'] == 'True' and
            ann['meta_anns']['Time']['value'] in ['Past','Recent']):
            doc_annotations.append(ann['cui'])
    single_doc = {doc: doc_annotations}
    relevant_annotations.update(single_doc)

del annotations

In [ ]:
parents_snomed_filter = [
    '38341003',  # Hypertension
    '73211009',  # diabetes
    '44054006',  # Diabetes type 2
    '46635009',  # Diabetes type 1
    '56265001',  # Heart disease
    '22298006',  # Prior heart attack/ MI   # question should this be infarction or ischemia?
    '414545008', # Ischemic heart disease
    '84114007',  # Heart failure
    '62914000',  # Cerebrovascular disease
    '230690007', # Stroke
    #'74732009',  # Mental health disease - Being Handled differently
    '111479008', #Organic mental disorder (disorder)
    '35489007',  # Depression
    '48694002',  # Anxiety
    '58214004',  # Schizophrenia
    #'33449004',  # Personality disorder - Grouped under Mental Health disease below
    '32709003',  # addiction
    '248279007', # frailty   # not inferred from text
    '400047006', # Peripheral vascular disease
    '14669001',  # AKI
    '709044004', # CKD
    '271809000', # peripheral oedema 
    '302226006', # peripheral neuropathy
    '397540003', # visual impairement 
    '82971005',  # need assisstance for mobility  # potential issues with this one. Not clear for explicit mentions
    '267036007', # Breathlessness
    '108241001' # dialysis procedure
]

In [ ]:
# Filter and column grouping
snomed_pt2ch_dict = {}
for sctid in parents_snomed_filter:
    snomed_pt2ch_dict[sctid] = get_all_children(sctid, cat.cdb.addl_info['pt2ch'])

In [ ]:
mental_health_snomed = ['444613000', #'Adult attention deficit hyperactivity disorder (disorder)'
                        '197480006', #'Anxiety disorder (disorder)'
                        '231538003', #'Behavioral and emotional disorder with onset in childhood (disorder)'
                        '128293007', #'Chronic mental disorder (disorder)'
                        '48500005', #'Delusional disorder (disorder)'
                        '129104009', #'Developmental mental disorder (disorder)'
                        '702711004', #'Mental disorder in mother complicating pregnancy (disorder)'
                        '199257008', #'Mental disorders during pregnancy, childbirth and the puerperium (disorder)'
                        '46206005', #'Mood disorder (disorder)'
                        '111479008', #'Organic mental disorder (disorder)'
                        '33449004', #'Personality disorder (disorder)'
                        '69322001' #'Psychotic disorder (disorder)'
                       ]

In [ ]:
# Filter and column grouping
mental_health_pt2ch_dict = {}
for sctid in mental_health_snomed:
    mental_health_pt2ch_dict[sctid] = get_all_children(sctid, cat.cdb.addl_info['pt2ch'])

### Split out Depressive Disorder, Schizophrenia and Bardet-Biedl Syndrome

In [ ]:
pop_codes = ['35489007', '58214004', '5619004']

In [ ]:
for k, v in mental_health_pt2ch_dict.items():
    for code in pop_codes:
        if code in v:
            v.remove(code)

In [ ]:
mental_health_dict = {}
mental_health_dict['74732009'] = []

for k, v in mental_health_pt2ch_dict.items():
    mental_health_dict['74732009'].extend(v)

del mental_health_pt2ch_dict

In [ ]:
def merge_two_dicts(x, y):
    """Given two dictionaries, merge them into a new dict as a shallow copy."""
    z = x.copy()
    z.update(y)
    return z

In [ ]:
full_pt2ch_dict = merge_two_dicts(snomed_pt2ch_dict, mental_health_dict)

In [ ]:
child_to_parent_dict = {}

for k, v in full_pt2ch_dict.items():
    for i in v:
        child_to_parent_dict.update({i: k})

#### Remove Additional Codes

In [ ]:
pop_cui_list = [
    '37151006',
    '53751009',
    '19092004',
    '46939000',
    '80387009',
    '20852007',
    '2829000',
    '773329005',
    '703389002',
    '193756007',
    '764455002',
    '720639008',
    '17827007',
    '41196008',
    '613003',
    '76129002',
    '717822006',
    '50643003',
    '232059000',
    '65207000',
    '770564004',
    '723410002',
    '68618008',
    '90099008',
    '51946000',
    '9527009',
    '17155009',
    '772127009',
    '1187122000',
    '40950007',
    '261195002',
    '39925003',
    '111307005',
    '609561005',
    '609570008',
    '445260006'
]

In [ ]:
for k in list(child_to_parent_dict.keys()):
    if k in pop_cui_list:
        del child_to_parent_dict[k]

### Struture Document Level Annotation Results

In [ ]:
refined_annotations = {}

for k, v in relevant_annotations.items():
    doc_list = []
    for i in v:
        try:
            doc_list.append(child_to_parent_dict[i])
        except:
            doc_list.append(i)
    refined_annotations.update({k:doc_list})

del relevant_annotations
del child_to_parent_dict
del parents_snomed_filter

In [ ]:
snomed_filter = [
    '38341003',  # Hypertension
    '73211009',  # diabetes
    '44054006',  # Diabetes type 2
    '46635009',  # Diabetes type 1
    '56265001',  # Heart disease
    '22298006',  # Prior heart attack/ MI   # question should this be infarction or ischemia?
    '414545008', # Ischemic heart disease
    '84114007',  # Heart failure
    '62914000',  # Cerebrovascular disease
    '230690007', # Stroke
    '74732009',  # Mental health disease   # check if right code
    '111479008', #Organic mental disorder (disorder)
    '35489007',  # Depression
    '48694002',  # Anxiety
    '58214004',  # Schizophrenia
    '5619004',   #Bardet-Biedl Syndrome
    #'33449004', # Personality disorder - Sits under Mental Health Disease
    '32709003',  # addiction
    '248279007', # frailty   # not inferred from text
    '400047006', # Peripheral vascular disease
    '14669001',  # AKI
    '709044004', # CKD
    '271809000', # peripheral oedema 
    '302226006', # peripheral neuropathy
    '397540003', # visual impairement 
    '82971005',  # need assisstance for mobility  # potential issues with this one. Not clear for explicit mentions
    '267036007', # Breathlessness
    '108241001', # dialysis procedure
    '4855003',   #Retinopathy due to diabetes mellitus (disorder)
    '19242006',  #Pulmonary oedema
    '371087003'  #Diabetic foot ulcer
    #,'302497006'#Hemodialysis - Sits under Diaylsis Procedure
]

In [ ]:
for s in snomed_filter:
    try:
        print(s, cat.cdb.cui2preferred_name[s])
    except:
        print(s, r'- Error')

In [ ]:
# Filter and column grouping
SCTID_groups_dict = {}
for sctid in snomed_filter:
    try:
        SCTID_groups_dict[cat.cdb.cui2preferred_name[sctid]] = [sctid]
    except:
        print(s, r'- Error')

In [ ]:
# invert snomed dict
def invert_dict(d): 
    inverse = dict() 
    for key in d: 
        # Go through the list that is saved in the dict:
        for item in d[key]:
            # Check if in the inverted dict the key exists
            if item not in inverse: 
                # If not create a new list
                inverse[item] = [key] 
            else: 
                inverse[item].append(key) 
    return inverse

inverted_snomed_dict = invert_dict(SCTID_groups_dict)

In [ ]:
#convert sctid to top level filter terms
doc2_top_term = {}
for k, v in refined_annotations.items():
    parent_sctid = []
    try:
        counter_dict = {x:0 for x in SCTID_groups_dict.keys()}
        
        for sctid in v:
            parent_sctid.extend(inverted_snomed_dict[sctid])
        for code in parent_sctid:
            counter_dict[code] += 1
    
        doc2_top_term[k] = counter_dict
        
    except KeyError: # pass codes which are not in the snomed relevant terms
        pass

In [ ]:
df_medcat_extract = pd.DataFrame.from_dict(doc2_top_term).T

In [ ]:
df_medcat_extract = df_medcat_extract.reset_index(drop=False).rename(columns={'index':'_id'})
df_medcat_extract['sum'] = df_medcat_extract.iloc[:,1:].sum(axis=1)

df_medcat_extract.head()

#### Save Base Output

In [ ]:
df_medcat_extract.to_csv(os.path.join(data_path, 'raw_data', '20251208_document_level_model_output.csv'), index=False)

## Import Document Meta-Data & Document Level Model Results

In [ ]:
medcat_extract_df = pd.read_csv(os.path.join(data_path, 'raw_data', '20251208_document_level_model_output.csv'))

In [ ]:
medcat_extract_df.head()

In [ ]:
clinical_noting_path = os.path.join(data_path, 'raw_data', 'elasticsearch_search_hits', '20250714_clinical_noting_comorbidities_search.csv')

clinic_notes_base = pd.read_csv(clinical_noting_path, dtype=object)

del clinical_noting_path

In [ ]:
clinical_noting_path = os.path.join(data_path, 'raw_data', 'elasticsearch_search_hits', '20251120_clinical_noting_comorbidities_search.csv')

clinic_notes_upd = pd.read_csv(clinical_noting_path, dtype=object)

del clinical_noting_path

In [ ]:
clinical_noting_path = os.path.join(data_path, 'raw_data', 'elasticsearch_search_hits', '20260202_clinical_noting_comorbidities_search.csv')

clinic_notes_latest = pd.read_csv(clinical_noting_path, dtype=object)

del clinical_noting_path

In [ ]:
epr_letters_path = os.path.join(data_path, 'raw_data', 'elasticsearch_search_hits', '20250714_epr_comorbidities_search.csv')

epr_letters_base = pd.read_csv(epr_letters_path, dtype=object)

del epr_letters_path

In [ ]:
epr_letters_path = os.path.join(data_path, 'raw_data', 'elasticsearch_search_hits', '20251120_epr_comorbidities_search.csv')

epr_letters_upd = pd.read_csv(epr_letters_path, dtype=object)

del epr_letters_path

In [ ]:
epr_letters_path = os.path.join(data_path, 'raw_data', 'elasticsearch_search_hits', '20260202_epr_comorbidities_search.csv')

epr_letters_latest = pd.read_csv(epr_letters_path, dtype=object)

del epr_letters_path

In [ ]:
epic_notes_path = os.path.join(data_path, 'raw_data', 'elasticsearch_search_hits', '20250714_epic_comorbidities_search.csv')

epic_notes_base = pd.read_csv(epic_notes_path, dtype=object)

del epic_notes_path

In [ ]:
epic_notes_path = os.path.join(data_path, 'raw_data', 'elasticsearch_search_hits', '20251120_epic_comorbidities_search.csv')

epic_notes_upd = pd.read_csv(epic_notes_path, dtype=object)

del epic_notes_path

In [ ]:
epic_notes_path = os.path.join(data_path, 'raw_data', 'elasticsearch_search_hits', '20260202_epic_comorbidities_search.csv')

epic_notes_latest = pd.read_csv(epic_notes_path, dtype=object)

del epic_notes_path

In [ ]:
ckd_docs_df = pd.concat([clinic_notes_base, epr_letters_base, epic_notes_base,
                         clinic_notes_upd, epr_letters_upd, epic_notes_upd,
                         clinic_notes_latest, epr_letters_latest, epic_notes_latest
                        ])

del clinic_notes_base, epr_letters_base, epic_notes_base, clinic_notes_upd, epr_letters_upd, epic_notes_upd, clinic_notes_latest, epr_letters_latest, epic_notes_latest

In [ ]:
adm_letter_filter = ckd_docs_df['document_Name'].str.contains('admin l', case=False, na=False)

In [ ]:
cols = ['patient_identifier4', 'patient_identifier2', 'patient_identifier3', 'document_Name', 'document_CreatedWhen', '_id']

ckd_docs_df = ckd_docs_df[(~adm_letter_filter)][cols].drop_duplicates().reset_index(drop=True)

ckd_docs_df['document_CreatedWhen'] = pd.to_datetime(ckd_docs_df['document_CreatedWhen'].apply(lambda x: str(x)[:10]), format='mixed').dt.date

del cols

In [ ]:
cols = ['master_person_id', 'patient_identifier3', 'patient_identifier4', 'patient_identifier2', 'inclusion_date', 'endpoint_date']

inclusion_patients_df = pd.read_csv(os.path.join(data_path, "opt_in_ckd_inclusion_patients.csv"))[cols]

del cols

inclusion_patients_df['inclusion_date'] = pd.to_datetime(inclusion_patients_df['inclusion_date']).dt.date
inclusion_patients_df['endpoint_date'] = pd.to_datetime(inclusion_patients_df['endpoint_date']).dt.date

In [ ]:
identifier3_dict = {}
for idx, row in enumerate(inclusion_patients_df.values):
    for nhs_num in row[1].split("'"):
        if len(nhs_num) <= 2:
            pass
        elif 'none' in nhs_num.lower():
            pass
        elif 'linked' in nhs_num.lower():
            pass
        else:
            identifier3_dict[nhs_num]=row[0]

ckd_docs_df['masterPersonId3'] = ckd_docs_df['patient_identifier3'].map(identifier3_dict)

In [ ]:
identifier4_dict = {}
for idx, row in enumerate(inclusion_patients_df.values):
    for nhs_num in row[2].split("'"):
        if len(nhs_num) <= 2:
            pass
        elif 'none' in nhs_num.lower():
            pass
        elif 'linked' in nhs_num.lower():
            pass
        else:
            identifier4_dict[nhs_num]=row[0]

ckd_docs_df['masterPersonId4'] = ckd_docs_df['patient_identifier4'].map(identifier4_dict)

In [ ]:
identifier2_dict = {}
for idx, row in enumerate(inclusion_patients_df.values):
    for nhs_num in row[3].split("'"):
        if len(nhs_num) <= 2:
            pass
        elif 'none' in nhs_num.lower():
            pass
        elif 'linked' in nhs_num.lower():
            pass
        else:
            identifier2_dict[nhs_num]=row[0]

ckd_docs_df['masterPersonId2'] = ckd_docs_df['patient_identifier2'].map(identifier2_dict)

In [ ]:
ckd_docs_df.insert(0, 'master_person_id', ckd_docs_df['masterPersonId3'].combine_first(ckd_docs_df['masterPersonId4']).combine_first(ckd_docs_df['masterPersonId2']))

ckd_docs_df = ckd_docs_df.merge(inclusion_patients_df[['master_person_id', 'inclusion_date', 'endpoint_date']], how='left', on='master_person_id')

ckd_docs_df = ckd_docs_df[['master_person_id', 'inclusion_date', 'endpoint_date', 'document_CreatedWhen', 'document_Name', '_id']]

inclusion_date_filter = (ckd_docs_df['document_CreatedWhen']>=ckd_docs_df['inclusion_date'])
endpoint_date_filter = (ckd_docs_df['document_CreatedWhen']<=ckd_docs_df['endpoint_date'])

ckd_docs_df = ckd_docs_df[endpoint_date_filter].reset_index(drop=True)

del identifier3_dict, identifier4_dict, identifier2_dict, inclusion_date_filter, endpoint_date_filter

ckd_docs_df.head()

### Combine Results

In [ ]:
joined_df = ckd_docs_df.merge(medcat_extract_df, how='left', on='_id') ## All Documents

for col in joined_df.columns[6:]:
    joined_df[col] = joined_df[col].apply(lambda x: 0 if pd.isna(x) else x)
    joined_df[col] = joined_df[col].astype(int)

del ckd_docs_df, medcat_extract_df

joined_df.head()

### Aggregate Results

In [ ]:
aggregator = {}
for c in joined_df.columns[6:]:
    aggregator.update({c: "sum"})

aggregator.update({'_id': list})

full_details_df = joined_df.groupby(['master_person_id']).agg(aggregator).reset_index()

full_details_df.head()

In [ ]:
cols = ['master_person_id', '_id', 'Number_of_Docs', 'sum', 'Hypertensive disorder, systemic arterial', 'Diabetes mellitus',
        'Diabetes mellitus type 2', 'Diabetes mellitus type 1', 'Heart disease', 'Myocardial infarction', 'Ischemic heart disease', 'Heart failure', 'Cerebrovascular disease',
        'Cerebrovascular accident', 'Mental disorder', 'Organic mental disorder', 'Depressive disorder', 'Anxiety', 'Schizophrenia', 'Bardet-Biedl syndrome', 'Addiction',
        'Frailty', 'Peripheral vascular disease', 'Acute renal failure syndrome', 'Chronic kidney disease', 'Peripheral edema', 'Peripheral nerve disease', 'Visual impairment',
        'Impaired mobility', 'Dyspnea', 'Dialysis procedure', 'Retinopathy due to diabetes mellitus', 'Pulmonary edema', 'Ulcer of foot due to diabetes mellitus']

def extract_contents(lst):
    return ', '.join(map(str, lst))

full_details_df['Number_of_Docs'] = full_details_df['_id'].apply(lambda x: len(x))

full_details_df['_id'] = full_details_df['_id'].apply(extract_contents)

full_details_df = full_details_df[cols]
full_details_df = full_details_df.drop_duplicates()

In [ ]:
for col in full_details_df.columns[2:]:
    full_details_df[col] = full_details_df[col].astype(int)

### Combine Final Results

In [ ]:
def diabetes_adder(row):
    result = row['Diabetes mellitus'] + row['Diabetes mellitus type 2'] + row['Diabetes mellitus type 1']

    return result

In [ ]:
full_details_df.insert(6, 'Diabetes_mellitus', full_details_df.apply(diabetes_adder, axis=1))

full_details_df = full_details_df.drop(columns=['Diabetes mellitus', 'Diabetes mellitus type 2', 'Diabetes mellitus type 1'])

In [ ]:
rename_dict = {}

for col in full_details_df.columns[2:]:
    rename_dict[col] = col.replace(',', '').replace(' ', '_')

full_details_df = full_details_df.rename(columns=rename_dict)

full_details_df.head()

In [ ]:
total_inc_pats = inclusion_patients_df['master_person_id'].nunique()
medcat_pats = full_details_df[full_details_df['sum']>0]['master_person_id'].nunique()
non_medcat_pats = total_inc_pats-medcat_pats

In [ ]:
print(f"Total Number of Patients Meeting the Inclusion Criteria: {total_inc_pats:,}")
print(f"Number of Patients with at least 1 Comorbidity: {medcat_pats:,}")
print(f"Number of Patients with No Comorbidities: {non_medcat_pats:,}")

### Final Thresholding

In [ ]:
dummy_details_df = full_details_df.copy()

dummy_details_df = dummy_details_df.drop(columns=['_id', 'Number_of_Docs', 'sum'])

for col in dummy_details_df.columns[1:]:
    dummy_details_df[col] = dummy_details_df[col].apply(lambda x: 1 if x > 0 else 0)

dummy_details_df.head()

### Output Final Data Data

In [ ]:
dummy_details_df.to_csv(os.path.join(data_path, 'processed_data', '20260206_CKD_Comorbidites_Model_Results.csv'), index=False)

# Sandbox